# 2023 MEPS Medication Adherence Teaching Notebook

This notebook rebuilds the 2023 MEPS medication adherence workflow from the file grains up.

Project questions:

1. Which medication groups show lower refill continuity?
2. How do cost and condition burden relate to adherence?
3. Can we later predict future non-adherence for a patient-drug pair?


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 40)
sns.set_theme(style="whitegrid")


## Load only 2023 files

2023 files used here:

- `h248a.xlsx`: prescribed medicines
- `h251.xlsx`: full-year consolidated person file
- `h248if1.xlsx`: condition-event linkage file
- `h249.xlsx`: medical conditions

The notebook uses relative paths so it can run from the repository root or from the notebook folder.


In [ ]:
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "MEPS" / "excels").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

MEPS_DIR = PROJECT_ROOT / "data" / "MEPS" / "excels"

rx_path = MEPS_DIR / "h248a.xlsx"
person_path = MEPS_DIR / "h251.xlsx"
clnk_path = MEPS_DIR / "h248if1.xlsx"
condition_path = MEPS_DIR / "h249.xlsx"
old_pivot_path = MEPS_DIR / "complete_df_pivot.xlsx"

for path in [rx_path, person_path, clnk_path, condition_path]:
    print(path.name, path.exists())


## File grains

Before merging, identify what one row means in each file.

- `h248a`: one prescription fill/refill/acquisition row.
- `h251`: one person row.
- `h248if1`: links events to conditions. For prescriptions, `EVENTYPE == 8`.
- `h249`: one reported medical condition row.

The Q1 adherence table starts from `h248a`, not from conditions.


In [ ]:
RX_COLUMNS = [
    "DUPERSID", "DRUGIDX", "RXRECIDX", "LINKIDX", "PANEL", "PURCHRD",
    "RXBEGMM", "RXBEGYRX", "RXNAME", "RXDRGNAM", "RXNDC",
    "RXQUANTY", "RXSTRENG", "RXDAYSUP",
    "TC1", "TC1S1", "TC1S1_1", "TC1S1_2",
    "RXSF23X", "RXXP23X", "PERWT23F", "VARSTR", "VARPSU",
]

PERSON_COLUMNS = [
    "DUPERSID", "AGE23X", "SEX", "RACETHX", "POVCAT23", "INSCOV23", "REGION23",
    "AFRDPM42", "DLAYPM42", "PMEDUP31", "PMEDUP42", "PMEDUP53",
    "DIABDX_M18", "HIBPDX", "CHDDX", "ANGIDX", "MIDX", "OHRTDX", "STRKDX",
    "ARTHDX", "ASTHDX", "ADHDADDX", "K6SUM42", "PHQ242", "ADPAIN42",
    "RXTOT23", "PERWT23F", "VARSTR", "VARPSU",
]

CONDITION_COLUMNS = [
    "DUPERSID", "CONDIDX", "AGEDIAG", "ICD10CDX", "CCSR1X", "RXCOND",
]

rx = pd.read_excel(rx_path, sheet_name="H248A", usecols=RX_COLUMNS, engine="calamine")
person = pd.read_excel(person_path, sheet_name="H251", usecols=PERSON_COLUMNS, engine="calamine")
clnk = pd.read_excel(clnk_path, sheet_name="H248IF1", engine="calamine")
condition = pd.read_excel(condition_path, sheet_name="H249", usecols=CONDITION_COLUMNS, engine="calamine")

print("rx rows:", len(rx), "unique people:", rx["DUPERSID"].nunique(), "unique patient-drugs:", rx["DRUGIDX"].nunique())
print("person rows:", len(person), "unique people:", person["DUPERSID"].nunique())
print("clnk rows:", len(clnk))
print("condition rows:", len(condition), "unique conditions:", condition["CONDIDX"].nunique())


In [ ]:
file_grain_check = pd.DataFrame({
    "file": ["h248a rx", "h251 person", "h248if1 clnk", "h249 condition"],
    "rows": [len(rx), len(person), len(clnk), len(condition)],
    "main_id": ["RXRECIDX", "DUPERSID", "CLNKIDX", "CONDIDX"],
    "unique_main_id": [
        rx["RXRECIDX"].nunique(),
        person["DUPERSID"].nunique(),
        clnk["CLNKIDX"].nunique(),
        condition["CONDIDX"].nunique(),
    ],
})

file_grain_check


## Inspect RXDAYSUP before calculating adherence

`RXDAYSUP` is the days supplied field. It has real day counts, but it also has MEPS special codes.

For adherence, use only `1 <= RXDAYSUP <= 990` as real supply days. Treat `999` as as-needed medication, not 999 days. Treat negative values as missing/special codes.


In [ ]:
rx["RXDAYSUP"].value_counts(dropna=False).head(15)


In [ ]:
rx_days_check = pd.Series({
    "all_rx_rows": len(rx),
    "valid_day_rows_1_to_990": rx["RXDAYSUP"].between(1, 990).sum(),
    "as_needed_999_rows": (rx["RXDAYSUP"] == 999).sum(),
    "missing_or_special_code_rows": (~rx["RXDAYSUP"].between(1, 990) & (rx["RXDAYSUP"] != 999)).sum(),
})

rx_days_check


In [ ]:
rx = rx.copy()

rx["valid_days_supplied"] = rx["RXDAYSUP"].where(rx["RXDAYSUP"].between(1, 990))
rx["is_prn_fill"] = rx["RXDAYSUP"].eq(999)
rx["has_missing_days_code"] = ~rx["RXDAYSUP"].between(1, 990) & ~rx["RXDAYSUP"].eq(999)

rx[["RXDAYSUP", "valid_days_supplied", "is_prn_fill", "has_missing_days_code"]].head()


## Q1: Which medication groups show lower refill continuity?

For Q1, start from prescriptions only.

Patient-drug key:

```text
DUPERSID + DRUGIDX
```

This avoids condition duplication. We add conditions later only after the patient-drug adherence table exists.


## Build patient-drug table

This table has one row per person-drug pair in 2023. It keeps data-quality flags so we can see how much of each row came from real days supplied versus special codes.


In [ ]:
patient_drug_key = ["DUPERSID", "DRUGIDX"]

first_drug_label = (
    rx.sort_values(patient_drug_key + ["RXRECIDX"])
    .drop_duplicates(patient_drug_key)
    [patient_drug_key + ["RXNAME", "RXDRGNAM", "RXNDC", "TC1", "TC1S1", "TC1S1_1", "TC1S1_2"]]
)

all_fill_counts = (
    rx.groupby(patient_drug_key)
    .agg(
        all_fill_rows=("RXRECIDX", "count"),
        prn_fill_count=("is_prn_fill", "sum"),
        missing_days_count=("has_missing_days_code", "sum"),
    )
    .reset_index()
)

valid_day_sums = (
    rx[rx["valid_days_supplied"].notna()]
    .groupby(patient_drug_key)
    .agg(
        valid_fill_count=("RXRECIDX", "count"),
        total_valid_days=("valid_days_supplied", "sum"),
    )
    .reset_index()
)

patient_drug = first_drug_label.merge(all_fill_counts, on=patient_drug_key, how="left")
patient_drug = patient_drug.merge(valid_day_sums, on=patient_drug_key, how="left")

patient_drug["valid_fill_count"] = patient_drug["valid_fill_count"].fillna(0).astype(int)
patient_drug["total_valid_days"] = patient_drug["total_valid_days"].fillna(0)

patient_drug.head()


In [ ]:
patient_drug[["all_fill_rows", "valid_fill_count", "total_valid_days", "prn_fill_count", "missing_days_count"]].describe()


## Build denominators

Two denominator choices are shown.

1. `coverage_365`: simple annual coverage using 365 days.
2. `coverage_from_start`: sensitivity version using medication start month/year when MEPS has it.

If the medication started before 2023, use 365. If it started during 2023 and month is valid, use days from that month through December 31. If start information is missing or unusable, use 365 and keep a flag.


In [ ]:
rx_start = rx.copy()
rx_start["valid_start_year"] = rx_start["RXBEGYRX"].where(rx_start["RXBEGYRX"].between(1900, 2023))
rx_start["valid_start_month"] = rx_start["RXBEGMM"].where(rx_start["RXBEGMM"].between(1, 12))

start_info = (
    rx_start.groupby(patient_drug_key)
    .agg(
        med_start_year=("valid_start_year", "min"),
        med_start_month=("valid_start_month", "min"),
    )
    .reset_index()
)

patient_drug = patient_drug.merge(start_info, on=patient_drug_key, how="left")


def days_from_month_to_year_end(month):
    start_date = pd.Timestamp(year=2023, month=int(month), day=1)
    end_date = pd.Timestamp(year=2023, month=12, day=31)
    return (end_date - start_date).days + 1


def choose_denominator(row):
    year = row["med_start_year"]
    month = row["med_start_month"]

    if pd.isna(year):
        return 365
    if year < 2023:
        return 365
    if year == 2023 and pd.notna(month):
        return days_from_month_to_year_end(month)
    return 365


def denominator_reason(row):
    year = row["med_start_year"]
    month = row["med_start_month"]

    if pd.isna(year):
        return "missing start year: used 365"
    if year < 2023:
        return "started before 2023: used 365"
    if year == 2023 and pd.notna(month):
        return "started in 2023: used days from start month"
    return "started in 2023 but month missing: used 365"

patient_drug["eligible_days_from_start"] = patient_drug.apply(choose_denominator, axis=1)
patient_drug["denominator_reason"] = patient_drug.apply(denominator_reason, axis=1)

patient_drug["coverage_365"] = patient_drug["total_valid_days"] / 365
patient_drug["coverage_from_start"] = patient_drug["total_valid_days"] / patient_drug["eligible_days_from_start"]
patient_drug["coverage_from_start_capped"] = patient_drug["coverage_from_start"].clip(upper=1)
patient_drug["below_75"] = patient_drug["coverage_from_start_capped"] < 0.75

patient_drug[["total_valid_days", "med_start_year", "med_start_month", "eligible_days_from_start", "coverage_from_start", "coverage_from_start_capped"]].head()


In [ ]:
patient_drug["denominator_reason"].value_counts(dropna=False)


## Medication group variable

The therapeutic class fields are coded numbers. For the first Q1 pass, use the most specific available class among `TC1S1_1`, `TC1S1`, and `TC1`.

Drug names are kept for drill-down, but the main answer should be by medication group.


In [ ]:
def make_med_group(row):
    for column in ["TC1S1_1", "TC1S1", "TC1"]:
        value = row[column]
        if pd.notna(value) and value > 0:
            return column + "=" + str(int(value))
    return "unclassified"

patient_drug["med_group"] = patient_drug.apply(make_med_group, axis=1)

q1_patient_drug = patient_drug[
    (patient_drug["total_valid_days"] > 0)
    & (patient_drug["prn_fill_count"] == 0)
].copy()

print("patient-drug rows before Q1 filter:", len(patient_drug))
print("patient-drug rows used for Q1:", len(q1_patient_drug))

q1_patient_drug[["DUPERSID", "DRUGIDX", "RXDRGNAM", "med_group", "total_valid_days", "coverage_from_start_capped", "below_75"]].head()


In [ ]:
class_summary = (
    q1_patient_drug.groupby("med_group")
    .agg(
        patient_drug_count=("DRUGIDX", "count"),
        person_count=("DUPERSID", "nunique"),
        median_total_days=("total_valid_days", "median"),
        median_valid_fills=("valid_fill_count", "median"),
        median_coverage=("coverage_from_start_capped", "median"),
        low_continuity_share=("below_75", "mean"),
    )
    .reset_index()
)

class_summary = class_summary[class_summary["patient_drug_count"] >= 30]
class_summary = class_summary.sort_values(
    ["low_continuity_share", "patient_drug_count"],
    ascending=[False, False],
)

class_summary.head(20)


In [ ]:
plt.figure(figsize=(10, 6))
plot_data = class_summary.head(15).sort_values("low_continuity_share")
sns.barplot(data=plot_data, x="low_continuity_share", y="med_group", color="#4C78A8")
plt.xlabel("Share below 75% coverage")
plt.ylabel("Medication group")
plt.title("Medication groups with lower refill continuity, 2023")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()


In [ ]:
drug_summary = (
    q1_patient_drug.groupby("RXDRGNAM")
    .agg(
        patient_drug_count=("DRUGIDX", "count"),
        person_count=("DUPERSID", "nunique"),
        median_total_days=("total_valid_days", "median"),
        median_coverage=("coverage_from_start_capped", "median"),
        low_continuity_share=("below_75", "mean"),
    )
    .reset_index()
)

drug_summary = drug_summary[drug_summary["patient_drug_count"] >= 30]
drug_summary.sort_values(["low_continuity_share", "patient_drug_count"], ascending=[False, False]).head(20)


## Add person variables

Now add demographics, insurance, and cost variables from `h251`. This is still one row per patient-drug.

These variables are for Q2a and later modeling. They should not change the Q1 medication totals.


In [ ]:
q1_with_person = q1_patient_drug.merge(person, on="DUPERSID", how="left", validate="many_to_one")

print("Q1 rows before person merge:", len(q1_patient_drug))
print("Q1 rows after person merge:", len(q1_with_person))
print("Rows missing AGE23X after merge:", q1_with_person["AGE23X"].isna().sum())

q1_with_person.head()


In [ ]:
cost_summary = (
    q1_with_person[q1_with_person["AFRDPM42"].isin([1, 2])]
    .groupby("AFRDPM42")
    .agg(
        patient_drug_count=("DRUGIDX", "count"),
        median_coverage=("coverage_from_start_capped", "median"),
        low_continuity_share=("below_75", "mean"),
    )
    .reset_index()
)

cost_summary["AFRDPM42_label"] = cost_summary["AFRDPM42"].map({1: "could not afford prescriptions", 2: "no affordability problem"})
cost_summary


## Q2b: Add condition linkage carefully

Conditions do not join directly to prescriptions on `DUPERSID`.

Correct path:

```text
h248a.LINKIDX -> h248if1.EVNTIDX -> h248if1.CONDIDX -> h249.CONDIDX
```

This section builds condition context after the patient-drug adherence table already exists.


In [ ]:
rx_event_links = rx[patient_drug_key + ["LINKIDX"]].drop_duplicates()

prescription_links = clnk[clnk["EVENTYPE"] == 8][["EVNTIDX", "CONDIDX"]].drop_duplicates()

rx_to_condition = rx_event_links.merge(
    prescription_links,
    left_on="LINKIDX",
    right_on="EVNTIDX",
    how="left",
)

condition_for_link = condition.drop(columns=["DUPERSID"])

rx_to_condition = rx_to_condition.merge(
    condition_for_link,
    on="CONDIDX",
    how="left",
)

print("rx event links:", len(rx_event_links))
print("linked rows after CLNK/condition merge:", len(rx_to_condition))
print("events with no linked condition:", rx_to_condition["CONDIDX"].isna().sum())

rx_to_condition.head()



In [ ]:
def short_code_list(values):
    clean_values = values.dropna().astype(str)
    clean_values = clean_values[~clean_values.isin(["-1", "-7", "-8", "-9", "-15"])]
    unique_values = sorted(clean_values.unique())
    return ", ".join(unique_values[:8])

condition_context = (
    rx_to_condition.groupby(patient_drug_key)
    .agg(
        linked_condition_count=("CONDIDX", "nunique"),
        linked_icd10_codes=("ICD10CDX", short_code_list),
    )
    .reset_index()
)

q2b_patient_drug = q1_with_person.merge(condition_context, on=patient_drug_key, how="left")
q2b_patient_drug["linked_condition_count"] = q2b_patient_drug["linked_condition_count"].fillna(0).astype(int)

q2b_patient_drug[["DUPERSID", "DRUGIDX", "RXDRGNAM", "linked_condition_count", "linked_icd10_codes"]].head()


## Age of diagnosis warning

`AGEDIAG` belongs to condition rows, not medication rows. It is often missing or inapplicable for old/chronic conditions. Do not use it as the denominator for medication adherence.

Use medication start month/year (`RXBEGMM`, `RXBEGYRX`) for the denominator sensitivity check instead.


In [ ]:
condition["AGEDIAG"].value_counts(dropna=False).head(15)


## Check the old pivot artifact without using it

This is a diagnostic only. The old pivot should not be the base for Q1 because it has already mixed medication, condition, and year logic.


In [ ]:
if old_pivot_path.exists():
    old_pivot = pd.read_excel(old_pivot_path, engine="calamine")
    print("old pivot rows:", len(old_pivot))
    print("old pivot columns:", list(old_pivot.columns))
    display(old_pivot[["DUPERSID", "DRUGIDX", "ICD10CDX", "meps_year", "RXDAYSUP", "AGEDIAG"]].head())
    display(old_pivot["RXDAYSUP"].describe())
    display(
        old_pivot.groupby(["DUPERSID", "DRUGIDX", "meps_year"])
        .size()
        .sort_values(ascending=False)
        .head(10)
    )
else:
    print("complete_df_pivot.xlsx not found")


## What Q1 answers and what it does not answer

Q1 can answer: which 2023 medication groups have lower annual supply coverage among patient-drug pairs.

Q1 does not answer exact refill gaps, because MEPS does not give a clean pharmacy fill calendar. It also does not prove the linked condition caused the drug. The CLNK file gives the best available condition-event linkage, but condition analysis belongs after the Rx adherence table is built.

Next steps in this notebook:

- Review the medication groups with the highest `low_continuity_share`.
- Decide whether to exclude supplements, as-needed drugs, and unclassified therapeutic classes.
- Use `q1_with_person` for cost and demographic summaries.
- Use `q2b_patient_drug` for condition burden summaries, not for rebuilding Q1 from scratch.
